# S03 — LLM Fundamentals: Tokens, Inference, and the Request/Response Lifecycle

**AI-Focused Forward Deployed Engineering · Session 03 · ~90 minutes (required path)**
**Environment:** Google Colab (GPU optional, CPU works) or local Jupyter with the same
packages. **Case framing:** generic model mechanics, with one Northstar-framed FDE
decision at the end (§9) connecting what you've just seen to a real customer ask.

S00, S01, and S02 deliberately treat "the model" as a black box: something that receives
a prompt and returns text or a structured decision. That's the right level of abstraction
for *most* FDE engineering decisions — but not all of them. When a customer asks "why does
the same ticket cost more to classify some days than others," or "can we just stream the
whole response instead of waiting," the answer lives one layer below that black box. This
session opens the box just far enough to answer those questions with mechanism, not
hand-waving — and no further than that.

## 🎯 What you will learn

By the end of this session's **required path**, you should be able to:

1. Explain tokenization and token IDs — and why a tokenizer is not an LLM.
2. Trace a prompt through model inference, end to end.
3. Explain, at an intuitive level, what embeddings and logits are and how a next token
   gets selected — without needing the underlying math.
4. Distinguish inference from training.
5. Explain how a context window constrains an application, in cost and behavior terms.
6. Explain sampling and generation settings (temperature, greedy vs. sampled decoding).
7. Compare streaming and non-streaming delivery — and what each one actually changes.
8. Distinguish a core LLM from an inference/serving runtime from a provider API.
9. Connect each of the above to a concrete engineering or customer-facing decision.

## 🗺️ How this notebook is organized

Every required section follows the same shape: **Concept → Intuition → Demonstration →
Observation → Engineering implication.** You see the idea, build a mental model for it,
run real code against a real (small) model, look at what actually happened, and then name
the decision that mechanism should change for you as an engineer.

Sections marked **(optional)** go deeper than the required outcomes above need. They're
real, working, and worth your time later — just not required to hit today's nine outcomes
in 90 minutes. Each one is clearly marked so you (or your instructor) can skip ahead
without losing the thread.

> [!NOTE]
> This is a **mechanics** session, not an **orchestration** or **retrieval** session. It
> does not build a tool-using agent (see **S01: Verified Agent Loops**) and does not build
> a retrieval pipeline (a future **Context Engineering and RAG** session will). Everything
> here happens *inside one model response*.

## Session roadmap (required path, ~90 minutes)

| # | Section | Time |
|---|---|---:|
| 0 | Setup | 3 min |
| 1 | Tokens & tokenization | 11 min |
| 2 | Loading the LLM + the full request path | 9 min |
| 3 | Embeddings — the intuition | 6 min |
| 4 | Logits & token selection | 8 min |
| 5 | Inference vs. training + the next-token generation loop | 15 min |
| 6 | Context window | 7 min |
| 7 | Streaming vs. non-streaming | 7 min |
| 8 | Three layers: LLM vs. inference runtime vs. provider API | 6 min |
| 9 | FDE decision: Northstar's token/cost budget | 7 min |
| 10 | Recap, checkpoints, transition | 4 min |
| | **Required total** | **~83 min** (+ buffer) |

Everything else in this notebook is marked **(optional)** and sits outside that budget.


## 0. 🧭 Setup

**Recommended (Colab):** `Runtime → Change runtime type → T4 GPU`, if one is available.
CPU also works for the small model used here — generation is just slower. Runtime
availability depends on your Colab plan.

> [!CAUTION]
> Never put a real API key in a shared or committed notebook. Upload your own `.env` only
> into **your own** Colab session — the session filesystem is temporary, but still
> readable by any code running in that session while it's open. The cell below never
> prints secret values.

Everything in the **required path** runs with no API keys and no network access beyond
the one-time model download. Keys are only needed for the optional hosted-provider
section (§8b).


In [ ]:
%pip -q install "transformers>=4.45,<5" "accelerate>=0.26" "python-dotenv>=1.0" "openai>=1.50"


In [ ]:
import os, sys, time, threading
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer

# Defensive: some terminals/notebook kernels report a non-UTF-8 stdout encoding,
# which raises UnicodeEncodeError the moment this notebook prints an emoji or a
# tokenizer's multi-byte marker character (both happen in the required path below).
try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except (AttributeError, ValueError):
    pass  # stdout does not support reconfigure here -- safe to continue

print("PyTorch:", torch.__version__)
print("Compute device:", "GPU" if torch.cuda.is_available() else "CPU")
print("API keys needed so far? No")


> [!NOTE]
> **(optional)** Only run the next cell if you intend to demonstrate the hosted-provider
> section (§8b) later. Skip it entirely for the required path.


In [ ]:
# OPTIONAL: upload your .env into the Colab session (only needed for section 8b).
UPLOAD_ENV = False  # Change to True only when you're ready to demo section 8b.

if UPLOAD_ENV:
    from dotenv import load_dotenv
    from google.colab import files
    uploaded = files.upload()
    if ".env" not in uploaded:
        print("No .env uploaded; the optional hosted-provider section will be skipped.")
    else:
        load_dotenv(".env")
        print("Environment variables loaded (values not printed).")
else:
    print("Skipped — running locally instead of Colab? Call load_dotenv('.env') directly.")


## 1. 🧩 Tokens and tokenization

**Concept.** A **token** is the unit of text a model actually reads — a whole word, a word
fragment, punctuation, or whitespace, depending on the tokenizer. A **token ID** is just
the integer that stands in for that token. A **tokenizer** is the text ⇄ ID conversion
algorithm plus its learned vocabulary. **It is not the language model.** It can't generate
anything; it only translates.

**Intuition.** Think of a tokenizer the way you'd think of a serialization format — it's
the agreed-upon encoding between "text a person reads" and "integers a neural network can
compute with." Different models can ship different tokenizers, so the same sentence can
become a different number of tokens depending on which model you ask.


In [ ]:
MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
print("Loading tokenizer:", MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

examples = [
    "Hello world!",
    "unbelievable!",
    "New Delhi is in India.",
    "Python is fun \U0001F40D",
]
for text in examples:
    ids = tokenizer.encode(text, add_special_tokens=False)
    print(f"{text!r:30s} -> {len(ids):2d} tokens -> {ids}")


**Expected output:** one line per example, each showing a token count and ID list — note
that `"Python is fun \U0001F40D"` (with an emoji) and `"unbelievable!"` both take **more**
tokens than their visible word count would suggest.


In [ ]:
sample = "The quick brown fox jumps over the lazy dog."
ids = tokenizer.encode(sample, add_special_tokens=False)
print("Original text: ", sample)
print("Token IDs:     ", ids)
print("Token strings: ", tokenizer.convert_ids_to_tokens(ids))
print("Decoded text:  ", tokenizer.decode(ids))
assert tokenizer.decode(ids) == sample


**Observation.** Encoding and decoding this sentence round-trips exactly — the assertion
passes. That's the tokenizer's *entire* job: text → IDs → text, nothing more. It never
looked at "meaning"; it looked up vocabulary entries.

> [!TIP]
> **⚙️ Engineering implication:** token counts, not word counts or character counts,
> drive both your context-window budget and most providers' billing. "This ticket has 40
> words" tells you nothing reliable about what it costs to classify — you have to count
> tokens. We'll use this directly in §9's cost exercise.

### ✅ Checkpoint
Is a token always a whole word? Is a tokenizer itself an LLM? Why might an emoji or an
uncommon word cost more tokens than a common one?

<details>
<summary>Answer</summary>

No; no; because token boundaries come from the tokenizer's trained vocabulary, not from
natural word boundaries — common patterns get single tokens, rare ones get split into more
pieces.
</details>


> [!NOTE]
> **(optional) Compare with a different tokenizer.** `tiktoken` is a tokenizer library —
> not a model — that can count tokens using an OpenAI-style encoding, entirely offline, no
> API key required. Skip this cell if you're on the required path; it reinforces the point
> above without adding a new one.


In [ ]:
# OPTIONAL
%pip -q install tiktoken
import tiktoken

other_tokenizer = tiktoken.get_encoding("cl100k_base")
text = "Tokenization is surprisingly interesting!"
local_ids = tokenizer.encode(text, add_special_tokens=False)
openai_style_ids = other_tokenizer.encode(text)
print("SmolLM2 token count:     ", len(local_ids))
print("cl100k_base token count: ", len(openai_style_ids))
print("Counts differ because the two tokenizers have different vocabularies.")


## 2. 🧩 Loading the LLM, and the full prompt-to-response path

**Concept.** The tokenizer alone cannot answer anything — it has no trained weights. Now
we load the actual **language model**: a neural network whose parameters were trained to
predict the next token from context. This specific model has ~135 million parameters,
far smaller than a flagship chat model, but the core mechanism — predict the next token
from everything so far — is the same one those larger models use.

> [!NOTE]
> Strictly, "LLM" means *large* language model. This teaching model is small enough to be
> called a causal language model or "small language model" (SLM). We use it because it's
> cheap to load and fast enough to inspect live, not because it's representative of a
> production model's quality.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
model = model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded language model: {MODEL_ID}")
print(f"Parameters: {n_params:,}")
print(f"Device: {device}")


**Intuition — the whole path in one diagram.** Everything from here to §5 is a tour of
this single pipeline, one box at a time:

```mermaid
flowchart TD
    A[Your prompt] --> B[Chat formatting]
    B --> C["Tokenizer: text to IDs"]
    C --> D["Embedding lookup<br/>IDs to vectors"]
    D --> E[Transformer layers]
    E --> F["Output logits<br/>score per vocabulary token"]
    F --> G[Select next token ID]
    G --> H{Stop condition met?}
    H -->|No, repeat| D
    H -->|Yes| I["Tokenizer decode:<br/>IDs to text"]
    I --> J[Response delivered]

    classDef io fill:#2a78d6,color:#ffffff,stroke:#184f95
    classDef model fill:#eb6834,color:#ffffff,stroke:#b84420
    classDef decision fill:#fab219,color:#0b0b0b,stroke:#a8790f

    class A,J io
    class B,C,D,E,F,G,I model
    class H decision
```

> [!IMPORTANT]
> **The output is not the input pipeline run backward.** The model does not take a vector
> and reverse the embedding lookup. It produces **new** vocabulary scores, picks a **new**
> token ID, and only *then* does the tokenizer decode that ID into text. Tokenization and
> decoding are roughly inverse operations of each other; the *whole generation process* is
> not an inverse of anything.

**Two different uses of the word "loop," named once here so neither gets confused later:**
this notebook's loop (§5) is the **next-token generation loop inside one model response**.
A separate, *outer* concept — an **agent loop** that makes several independent model
requests and executes tools between them — is **S01: Verified Agent Loops**'s entire
subject, not this session's. One model response can contain many next-token iterations and
still be a single step of an agent loop, or the only step if there's no agent loop at all.


In [ ]:
user_question = "Explain a rainbow in one easy sentence."
messages = [
    {"role": "system", "content": "You are a helpful engineering assistant."},
    {"role": "user", "content": user_question},
]
formatted = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print("User originally typed:\n", user_question)
print("\nWhat the model is actually prompted with (chat-formatted text):\n")
print(formatted)


**Observation.** The model never sees your raw question — it sees a chat-formatted string
with role markers and special tokens the tokenizer and chat template agreed on in advance.

> [!TIP]
> **⚙️ Engineering implication:** when a response looks wrong, "is this a chat-template
> problem, a tokenizer problem, a model problem, or a decoding problem?" is a real
> debugging question with a real answer, because these are genuinely separate stages —
> not synonyms for "the AI." This is the same layer-boundary discipline **S02** applies at
> the application/API/model level; here you're seeing the layers *inside* a single call.


## 3. 🧩 Embeddings — the intuition, without the math

**Concept.** A token ID like `1234` is just a label — it does **not** mean that token is
"bigger" or "more important" than token `1233`. The model needs numbers it can actually
compute with, so it looks up a trained vector (a list of numbers) for every token ID. This
**embedding layer** lives *inside* the generative model — you don't call a separate
service for it.

**Intuition.** Think of it like a lookup table from "token ID" to "a point in a very
high-dimensional space," where nearby points tend to behave similarly during generation.
You don't need the geometry to use this system — you need to know the lookup exists and
that it's internal to the model, not a separate "embedding model" (those exist too, for a
different job — semantic search — which a future RAG session covers; not needed here).


In [ ]:
example_text = "What is AI?"
example_ids = tokenizer(example_text, return_tensors="pt")["input_ids"].to(device)
embedding_layer = model.get_input_embeddings()
with torch.inference_mode():
    input_vectors = embedding_layer(example_ids)

print("Original text:", example_text)
print("Token ID sequence:", example_ids[0].tolist())
print("Token ID tensor shape:", tuple(example_ids.shape))
print("Embedding tensor shape:", tuple(input_vectors.shape))


**Observation.** If the tokenizer produced, say, 5 tokens and the model uses 576-dimensional
embeddings, the shapes read `[1, 5]` (one prompt, five IDs) → `[1, 5, 576]` (one prompt,
five vectors, 576 numbers each) — **one vector per token**, not one vector for the whole
prompt. The transformer layers (not explored here — that's genuinely a deeper topic) then
let those per-token vectors influence each other based on context.

> [!TIP]
> **⚙️ Engineering implication:** this is why "the model's context" is measured in
> *tokens*, and why a longer prompt is strictly more compute, every time — there's no
> shortcut where extra tokens are "free" because they're similar to earlier ones.

> [!NOTE]
> **(optional) A sharper nuance.** The *initial* embedding lookup for a given token ID is
> always the same vector, regardless of neighboring words — it's only after the
> transformer layers process the sequence that a token's *contextualized* representation
> can differ by context. Skip this if you're short on time; it's a refinement, not a new
> required idea.


In [ ]:
# OPTIONAL
tid = int(example_ids[0, 0])
with torch.inference_mode():
    first_lookup = embedding_layer(torch.tensor([[tid]], device=device))[0, 0]
    second_lookup = embedding_layer(torch.tensor([[tid]], device=device))[0, 0]
print("Same token ID:", tid)
print("Same INITIAL embedding on repeated lookup:", torch.equal(first_lookup, second_lookup))
print("After the transformer layers, a token's representation can still depend on context.")


## 4. 🧩 Logits and token selection

**Concept.** After the transformer layers process the embedding vectors, a final
**output projection** turns the result into one score — a **logit** — for *every* token in
the vocabulary. A **softmax** turns those raw scores into a probability distribution; some
selection rule (greedy, or sampling — §5) then picks one actual next token ID.

**Intuition — is this the reverse of §3?** No. Section 3 was `token ID → vector`. This is
`vector → score for every possible next ID → pick one ID`. There's no step that reconstructs
the *original* input ID — the model is choosing a **new** one, informed by everything it's
seen so far.


In [ ]:
ids_for_output = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Answer briefly: What is a rainbow?"}],
    add_generation_prompt=True,
    return_tensors="pt"
).to(device)

with torch.inference_mode():
    result = model(ids_for_output)
    logits = result.logits[0, -1, :]
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, k=8)

print("Top 8 candidate next tokens:")
for p, tid in zip(top_probs.tolist(), top_ids.tolist()):
    print(f"  {p:6.2%}  id={tid:<6d} token={tokenizer.decode([tid])!r}")


**Observation.** These are candidates for the **single next token**, not a finished
sentence and not a semantic "understanding" of the question — some entries may look like
word fragments or stray punctuation, because that's the model's actual vocabulary unit,
not an English word.

> [!IMPORTANT]
> **⚙️ Engineering implication — and the single most important caution in this
> notebook:** when a model "explains its reasoning" in generated text, that explanation
> was produced by this exact same mechanism — next-token prediction over a vocabulary —
> not by inspecting its own internal computation and reporting on it faithfully. A
> generated rationale is a plausible-sounding **output**, evaluated the same way any other
> generated text is, not a trustworthy trace of *why* the model actually produced its
> answer. Treat a model's stated reasoning as a user-facing string to verify, never as
> ground truth about its internals — this is the same caution **S02** states about
> structured-output rationales, grounded here in the mechanism that makes it true.

> [!NOTE]
> **(optional)** Skip straight to §5 if you're on the required path — the softmax
> mechanics above are enough intuition for everything that follows.


## 5. 🧩💻 Inference vs. training, and the next-token generation loop

**Concept.** **Training** is the (separate, offline, expensive) process that adjusted this
model's weights so it predicts plausible next tokens. **Inference** is using those
already-fixed weights to make a prediction — nothing about the weights changes while you
run this notebook. Everything you do from here on is inference.

A call to `model.generate(...)` runs the repeated "predict one token, append it, repeat"
process for you automatically. First, a small convenience function we'll reuse:


In [ ]:
def local_generate(user_text, max_new_tokens=60, do_sample=False, temperature=0.8):
    chat = [
        {"role": "system", "content": "You are a helpful engineering assistant. Keep answers simple."},
        {"role": "user", "content": user_text},
    ]
    inputs = tokenizer.apply_chat_template(
        chat, add_generation_prompt=True, return_tensors="pt"
    ).to(device)
    kwargs = dict(max_new_tokens=max_new_tokens, do_sample=do_sample,
                  pad_token_id=tokenizer.eos_token_id)
    if do_sample:
        kwargs["temperature"] = temperature
    start = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(inputs, **kwargs)
    elapsed = time.perf_counter() - start
    new_tokens = output_ids[0][inputs.shape[-1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return text, len(new_tokens), elapsed


### 5a. See one next-token prediction directly

Before letting `model.generate(...)` run the whole loop for us, let's look at the loop's
**single step** in isolation — the same scores-over-the-vocabulary idea from §4, applied to
a different prompt.


In [ ]:
prompt_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": "The capital of France is"}],
    add_generation_prompt=True,
    return_tensors="pt"
).to(device)
with torch.inference_mode():
    outputs = model(prompt_ids)
    logits = outputs.logits[0, -1, :]
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, k=5)

for p, tid in zip(top_probs.tolist(), top_ids.tolist()):
    print(f"  {p:6.2%}  {tokenizer.decode([tid])!r}")


**Expected output:** the top candidate should be a plausible continuation like `" Paris"` —
this tiny model won't always get general knowledge right, and that's fine; we're watching
the *mechanism*, not grading the model.

### 5b. 🛠️ Watch the token-by-token loop yourself — the critical demonstration

`model.generate(...)` hides the step-by-step process behind one function call. Here, we
implement it ourselves, deliberately slowly, so you watch exactly what "autoregressive
generation" means: at every iteration, feed the current tokens in, pick the highest-scoring
next one, append it, and repeat.

> [!TIP]
> **💡 Think, before running this cell:** if this loop ran forever with no stop condition,
> what would eventually happen? Keep your answer in mind — it's exactly the question
> **S01: Verified Agent Loops** asks about its own (different) kind of loop.


In [ ]:
chat = [{"role": "user", "content": "Complete the sentence: A good teacher is"}]
current_ids = tokenizer.apply_chat_template(
    chat, add_generation_prompt=True, return_tensors="pt"
).to(device)
generated = []

for step in range(12):
    with torch.inference_mode():
        next_logits = model(current_ids).logits[:, -1, :]
    next_id = torch.argmax(next_logits, dim=-1, keepdim=True)
    current_ids = torch.cat([current_ids, next_id], dim=-1)
    piece = tokenizer.decode(next_id[0])
    generated.append(piece)
    print(f"step {step+1:2d}: picked {piece!r:15s}  running text so far: {''.join(generated)!r}")

print("\nFinal generated text:", "".join(generated))


**Observation.** Twelve iterations, one new token chosen each time, each decision made
using *only* the tokens so far (including the ones this same loop just generated). Real
inference engines cache earlier computations (the **KV cache**) so they don't redo the
whole prefix every step — we skipped that optimization on purpose, for clarity, not because
it doesn't matter in production.

> [!IMPORTANT]
> **⚙️ Engineering implication — generation loop vs. agent loop, stated precisely now
> that you've seen both halves of the distinction:**
> - **This loop** (what you just ran): many *next-token* iterations inside **one** model
>   response. You never write this loop yourself against a hosted API — the provider runs
>   it and returns (or streams) the finished text.
> - **An agent loop** (S01's subject, not built here): the *outer* process that can issue
>   several independent model **requests**, let the model request a tool, execute that
>   tool in application code, and feed the result back — each of those steps may itself
>   contain a full next-token loop like the one above. They're nested, not the same thing,
>   and conflating them is a common and costly confusion in production debugging.

### 5c. Sampling and temperature

Two different selection rules for "which token to pick next," both still one token at a
time:

- **Greedy** (`do_sample=False`): always pick the highest-scoring token — deterministic.
- **Sampling** (`do_sample=True`): pick randomly, weighted by the (temperature-adjusted)
  probabilities — not deterministic.
- **Temperature** reshapes those probabilities: lower is more predictable, higher is more
  varied (and can get worse, not just "more creative").


In [ ]:
question = "Write one short sentence about exploring space."
for label, sampling, temp in [
    ("Greedy", False, 1.0),
    ("Sampling (T=0.7)", True, 0.7),
    ("Sampling (T=1.2)", True, 1.2),
]:
    answer, _, _ = local_generate(question, max_new_tokens=32,
                                  do_sample=sampling, temperature=temp)
    print(f"{label}: {answer!r}")


**Observation.** Generation is still one token at a time in every row above — only the
*selection rule* changed. Re-run the cell and the greedy row stays identical; the sampled
rows can change.

> [!TIP]
> **⚙️ Engineering implication:** temperature is a **product decision**, not a correctness
> knob. A classification task (S02's `TicketDecision`) usually wants greedy, deterministic,
> repeatable output; a brainstorming or copy-drafting task might reasonably want some
> sampling. Pick it from the task's actual requirement for repeatability, not a default.

### ✅ Checkpoint
Does `model.generate(...)` do something fundamentally different from the manual loop you
just ran in §5b, or does it run the same kind of loop faster and with more bookkeeping?

<details>
<summary>Answer</summary>

The same kind of loop, with optimizations (KV caching, batching, stopping-criteria
handling) and convenience you'd otherwise have to write yourself — not a different
mechanism.
</details>


## 6. 🧩 Context window: what can the model see?

**Concept.** A model's **context window** is the maximum number of tokens it can take into
account for one request — your instructions, the conversation history, any documents you
supplied, and the tokens it's about to generate, all competing for the same budget.

**Intuition.** It's working memory for *this request only*, not a notebook the model keeps
for you across requests. If an application wants the model to "remember" something from
three turns ago, *the application* has to re-send it — nothing persists on the model's side
between calls.


In [ ]:
short_chat = [{"role": "user", "content": "Hi!"}]
long_chat = [
    {"role": "user", "content": "Explain photosynthesis."},
    {"role": "assistant", "content": "Plants use sunlight to help make food."},
    {"role": "user", "content": "Explain it again using an analogy for a child."},
]
for label, chat in [("Short", short_chat), ("More conversation", long_chat)]:
    token_ids = tokenizer.apply_chat_template(chat, add_generation_prompt=True)
    print(f"{label:20s} conversation: {len(token_ids)} input tokens")
print("\nContext window = a limited token budget, not unlimited memory.")


**Observation.** The longer conversation costs measurably more input tokens before the
model has generated a single word of its answer — and that cost grows with *every* turn
you keep resending, not just the newest one.

> [!IMPORTANT]
> **⚙️ Engineering implication:** "the model has a huge context window" is not, by itself,
> a reason to attach everything you have to every request. More tokens means more cost and
> more latency, and irrelevant material can crowd out the signal that actually matters.
> This is exactly the caution **S02** raises about attaching an entire knowledge base to
> every ticket — you're now seeing *why* it's true, in tokens, not just as a rule to follow.
> We'll put a number on it in §9.

### ✅ Checkpoint
If an application wants a model to "remember" a customer's name across five separate API
calls, where does that memory actually have to live?

<details>
<summary>Answer</summary>

In the application — it has to re-send the relevant prior context (or a summary of it) on
every call. The model itself has no persistent memory between separate requests.
</details>


## 7. 🧩 Streaming vs. non-streaming: same generation, different delivery

**Concept.** The model predicts tokens one at a time **regardless** of delivery mode.

- **Non-streaming:** the serving layer collects the whole result, then returns one
  completed response. Your application typically displays it only after the call finishes.
- **Streaming:** the serving layer sends chunks as they become available, so your
  application can display text progressively.

**Intuition.** Streaming is a UX and perceived-latency choice at the *delivery* layer — it
does not change how many tokens get generated, how much compute it takes, or (usually) the
total time until the last token exists.


In [ ]:
stream_question = "Explain why the sky looks blue in two simple sentences."

print("--- Non-streaming: printed only after generate() finishes ---")
start = time.perf_counter()
text, count, elapsed = local_generate(stream_question, max_new_tokens=70)
print(text)
print(f"{count} output tokens; {elapsed:.2f} s total, nothing shown until now")


In [ ]:
print("--- Streaming: display text chunks as soon as they're available ---")
chat = [{"role": "user", "content": stream_question}]
inputs = tokenizer.apply_chat_template(
    chat, add_generation_prompt=True, return_tensors="pt"
).to(device)
streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

def run_generation():
    with torch.inference_mode():
        model.generate(
            inputs, max_new_tokens=70, pad_token_id=tokenizer.eos_token_id,
            streamer=streamer,
        )

thread = threading.Thread(target=run_generation)
thread.start()
for chunk in streamer:
    print(chunk, end="", flush=True)
thread.join()
print()


**Observation.** Both cells generate the same kind of output from the same process — only
*when* you see pieces of it differs. Streaming did not make generation itself faster.

> [!TIP]
> **⚙️ Engineering implication:** choose streaming for its actual benefit — **perceived**
> responsiveness in a user-facing chat UI — not because it changes total compute or
> guarantees a faster end-to-end result. For a backend classification call like S01/S02's
> ticket triage, where nothing is displayed token-by-token to a human, streaming usually
> buys you nothing.

> [!NOTE]
> **(optional) TTFT / TPOT vocabulary.** **Time to first token** and **time per output
> token** are the two numbers people actually mean when they say "latency" for a streaming
> API. Precise measurement needs provider-reported timestamps, not a local text streamer's
> buffering behavior — treat the vocabulary as useful shorthand, not something this
> notebook measures rigorously.

### ✅ Checkpoint
Did streaming change *how* the model generates tokens, or *when* your application receives
them?

<details>
<summary>Answer</summary>

Only when they're received — the underlying token-by-token generation process (§5) is
identical either way.
</details>


## 8. 🧩 Three layers: LLM, inference runtime, and provider API

**Concept.** Everything you've run locally in this notebook — tokenizer, model, embedding
lookup, logits, the generation loop — is the **core LLM** plus the Python code standing in
for an **inference/serving runtime** (a real one, like vLLM, adds batching, the KV cache,
and request scheduling; we skipped those for clarity). A **provider API** wraps that
runtime with authentication, routing, rate-limiting, and response delivery (streamed or
not).

```mermaid
flowchart TB
    A[Client / your application] -->|prompt + settings| B["Provider API<br/>auth, routing, rate limits"]
    B --> C["Inference / serving runtime<br/>chat formatting, batching, scheduling"]
    C --> D["Core LLM<br/>tokenizer, embeddings, transformer, logits, selection"]
    D --> C
    C --> B
    B -->|complete or streamed response| A

    classDef io fill:#2a78d6,color:#ffffff,stroke:#184f95
    classDef infra fill:#eb6834,color:#ffffff,stroke:#b84420
    classDef core fill:#0ca30c,color:#ffffff,stroke:#076b07

    class A io
    class B,C infra
    class D core
```

> [!NOTE]
> **This nests inside S02's three-layer diagram** (`Northstar application → model-serving
> API/gateway → LLM`), not a competing one. S02's "model-serving API/gateway" box *is* the
> "provider API" + "inference runtime" boxes above, zoomed out to decision-making depth.
> Same boundary, two levels of detail — pick the one the question in front of you needs.

**Why the separation matters in practice:** a real provider may distribute these functions
across different services, and may add monitoring, caching, safety checks, and GPU
scheduling on top. `vLLM` is a real example of the "inference runtime" box — it's not an
LLM, not a tokenizer, and not an agent; it serves models (sometimes through an
OpenAI-compatible API). None of this is required to use this notebook — it's here because
it's exactly the kind of fact that turns "the AI is slow" into an actual diagnosis: is this
a network/API problem, a runtime/scheduling problem, or the model itself?


> [!TIP]
> **⚙️ Engineering implication:** when you call a hosted API, you never write the §5
> generation loop, never touch the §3 embedding layer, and never see §4's logits directly —
> the provider's inference runtime handles all of it internally. What you *can* usually
> control through API parameters: max output tokens, sampling settings, and streaming mode
> — the three things this notebook just showed you the mechanism behind.

> [!NOTE]
> **(optional) Hosted-provider comparison (§8b).** The cell below is **off by default** and
> requires API keys — it's not part of the required 90-minute path. It exists to show that
> the same request/response shape you just built locally is exactly what a hosted provider
> call looks like from the outside, with the generation loop itself hidden.


### 8b. 💻 Optional: one code pattern, two hosted providers

> [!CAUTION]
> Requires `OPENAI_API_KEY` and/or `OPENROUTER_API_KEY` set as environment variables (never
> pasted into the notebook). `openrouter/free` has rate limits and may select a different
> underlying model per request — "free to call" is not the same claim as "open-weight
> model." If no keys are present, the cells below skip instead of failing.


In [ ]:
# OPTIONAL — requires API keys; skips gracefully if absent.
from openai import OpenAI

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openrouter/free")

providers = {}
if os.getenv("OPENAI_API_KEY"):
    providers["OpenAI"] = (OpenAI(api_key=os.environ["OPENAI_API_KEY"]), OPENAI_MODEL)
if os.getenv("OPENROUTER_API_KEY"):
    providers["OpenRouter"] = (
        OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"]),
        OPENROUTER_MODEL,
    )
if not providers:
    print("No provider keys found — skipping the hosted-provider comparison.")
else:
    print("Providers available for this demo:", list(providers.keys()))


In [ ]:
# OPTIONAL
def provider_nonstream(client, model_name, prompt):
    started = time.perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100,
        stream=False,
    )
    elapsed = time.perf_counter() - started
    print("Returned response:", response.choices[0].message.content)
    print(f"Elapsed: {elapsed:.2f}s")
    if getattr(response, "usage", None):
        print("Reported usage:", response.usage)


def provider_stream(client, model_name, prompt):
    started = time.perf_counter()
    stream = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100,
        stream=True,
    )
    first_text_s = None
    chunk_count = 0
    for event in stream:
        delta = event.choices[0].delta.content if event.choices else None
        if delta:
            if first_text_s is None:
                first_text_s = time.perf_counter() - started
            chunk_count += 1
            print(delta, end="", flush=True)
    print()
    print(f"\nFirst visible chunk after ~{first_text_s:.2f}s; {chunk_count} chunks total")


for name, (client, model_name) in providers.items():
    print(f"=== {name} ({model_name}) — non-streaming ===")
    provider_nonstream(client, model_name, "Explain tokenization in one sentence.")
    print(f"\n=== {name} ({model_name}) — streaming ===")
    provider_stream(client, model_name, "Explain tokenization in one sentence.")
    print()


**(optional) Instructor note:** even with an identical prompt, the local model and any
hosted provider can produce different text, because they have different trained weights,
different tokenizers/chat templates, different inference settings, and (if sampling)
genuine randomness — none of that means anything is broken.


## 9. 🛠️ FDE decision: Northstar's token/cost budget

**The ask.** Northstar's support lead looked at last month's invoice and asked: *"Why does
classifying the exact same kind of ticket cost different amounts depending on which agent
submitted it?"* The honest answer is §1 and §6, applied: agents who paste in the full
conversation history (or several related tickets) before asking for a classification are
paying for every one of those extra tokens, every single call — not because the model is
inconsistent, but because token count, not "ticket complexity," drives both cost and the
context budget.


In [ ]:
def estimate_usd(input_tokens, output_tokens, input_usd_per_m, output_usd_per_m):
    return (input_tokens * input_usd_per_m + output_tokens * output_usd_per_m) / 1_000_000

# Hypothetical rates for this exercise only -- not a real provider's current pricing.
INPUT_RATE, OUTPUT_RATE = 1.00, 4.00

short_ticket_tokens = len(tokenizer.apply_chat_template(short_chat, add_generation_prompt=True))
long_ticket_tokens = len(tokenizer.apply_chat_template(long_chat, add_generation_prompt=True))

for label, in_tokens in [("Ticket text only", short_ticket_tokens),
                         ("Ticket + full prior conversation", long_ticket_tokens)]:
    cost = estimate_usd(in_tokens, 60, INPUT_RATE, OUTPUT_RATE)
    print(f"{label:35s}: {in_tokens:4d} input tokens -> ${cost:.6f} per call "
          f"-> ${cost * 10_000:.2f} per 10,000 tickets")


**Observation.** Same classification task, same model — the only variable is how much
history got attached, and the 10,000-ticket projection turns a small per-call difference
into a real monthly line item.

### ✍️ Your take-home artifact

Write the one-page note you'd actually send Northstar: state the finding, recommend a
concrete context-size policy for the ticket-triage system (e.g., "attach the current
ticket plus the last N exchanges, not the full thread"), and name the one trade-off your
policy accepts (e.g., occasionally losing older context that *would* have mattered).

_Write your note here._


## 10. ✅ Recap and transition

```mermaid
flowchart TB
    A[User text] --> B["Tokenizer<br/>not an LLM"]
    B --> C[Token IDs]
    C --> D["Embedding layer<br/>inside the LLM"]
    D --> E[Transformer layers]
    E --> F["Output logits<br/>scores per vocabulary token"]
    F --> G["Token selection<br/>greedy or sampled"]
    G -->|repeat until stop| D
    G --> H["Tokenizer decode<br/>IDs to text"]
    H --> I["Response delivery<br/>streamed or complete"]

    classDef io fill:#2a78d6,color:#ffffff,stroke:#184f95
    classDef notllm fill:#8a8f98,color:#ffffff,stroke:#5c6066
    classDef core fill:#0ca30c,color:#ffffff,stroke:#076b07

    class A,I io
    class B,H notllm
    class D,E,F,G core
```

**Three statements worth remembering:**

1. A **tokenizer is not an LLM** — it only converts text ⇄ token IDs.
2. Output is **not the input pipeline reversed** — the model produces *new* token IDs from
   logits; only decoding mirrors tokenization.
3. A provider's serving runtime normally runs the generation loop *for* you.
   **`stream=True` changes delivery, not the underlying next-token process.**

### ✅ Checkpoints (pick 2–3 if short on time)

1. Why can the same prompt cost different amounts on two different days, with nothing
   about the *task* having changed?
2. What's the difference between the loop you ran in §5b and an agent loop?
3. Why isn't a model's stated "reasoning" the same thing as its actual internal
   computation?
4. If a customer asks "just stream everything, it'll be faster," what's the accurate
   response?

<details>
<summary>Answers</summary>

1. More input or output tokens were used — most commonly from more conversation history
   being re-sent (§6, §9).
2. §5b's loop is next-token prediction inside one model response; an agent loop (S01) makes
   multiple independent model requests and executes tools between them.
3. Generated "reasoning" text is produced by the same next-token mechanism as any other
   output (§4) — it is not a readout of the model's internal computation.
4. Streaming changes when output becomes visible, not how much compute generation takes or
   how fast the last token is produced (§7).
</details>

### 🗺️ Where this goes next

This session opened the box on **one model call**. **S01: Verified Agent Loops** is about
what happens *around* several of those calls — tool execution, harnesses, and independent
verification. **S02: Structured Outputs and Model Selection** is about trusting a single
call's output enough to build a business process on it — typed contracts and task-first
model comparison. Neither depends on this session, and this session doesn't depend on
either of them; take them in whatever order suits you.

## 📚 References

- [SmolLM2-135M-Instruct model card](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct)
- [Hugging Face tokenizer documentation](https://huggingface.co/docs/transformers/main_classes/tokenizer)
- [Hugging Face generation documentation](https://huggingface.co/docs/transformers/main_classes/text_generation)
- [OpenAI Chat Completions API](https://platform.openai.com/docs/api-reference/chat)
- [vLLM documentation](https://docs.vllm.ai/)

---

## 📚 Optional self-study activities

Not required for the nine learning outcomes above — for learners who want more practice.

1. **Tokenizer exploration:** change punctuation, add an emoji, or switch language in §1's
   examples. Why aren't tokens always whole words?
2. **Predicting tokens:** in §5a, change the prompt to `"2 + 2 ="` or `"Once upon a
   time"`. Are the printed probabilities for a complete sentence, or only the next token?
3. **Draw the journey from memory:** without looking back, sketch
   `prompt → tokenizer → embeddings → transformer → logits → selection → decode → response`,
   then add branches for streaming and non-streaming.
4. **The embedding question:** in §3's demo, change `example_text` to a longer sentence.
   Predict which dimension of the embedding tensor grows before you run it.
5. **Provider response modes:** in §8b, confirm which parameter changes delivery mode, and
   convince yourself it doesn't change the underlying generation loop.

**Execution note:** offline/local-model cells were exercised when this notebook was built;
the optional hosted-provider cells (§8b) were deliberately left unexecuted and unverified
against any specific provider account.
